In [1]:
!unzip -q tika_data.zip -d /content/


replace /content/parser/AbstractEncodingDetectorParser.java? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [2]:
!pip install -U transformers accelerate bitsandbytes sentencepiece scikit-learn

In [7]:
import os
import torch
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from google.colab import userdata
from transformers import BitsAndBytesConfig

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [9]:

filtered_rsf = Path("/content/filtered_rsf.rsf")
source_files = Path("/content")
output = Path("/content")

embedding_model = "nomic-ai/nomic-embed-code"
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

def get_classes(rsf_path):
    """Reads the filtered .rsf file to fetch all Java classes present in the repository."""
    classes = set()
    with open(rsf_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                classes.add(parts[1])
                classes.add(parts[2])
    return sorted(list(classes))

def get_source_files(full_filename, source):
    """Locates the .java file or returns an empty stub if missing."""
    filename = full_filename.split(".")[-1].split("$")[0] + ".java"

    for x in source.rglob(filename):
        if x.is_file():
            return x.read_text(errors="replace")

    return f"public class {full_filename.split('.')[-1].split('$')[0]} {{}}"


def build_structural_matrix(rsf_path, class_list):
    """Builds and normalizes the structural dependency matrix."""
    n = len(class_list)
    class_idx = {cls_name: i for i, cls_name in enumerate(class_list)}
    struct_mat = np.zeros((n, n), dtype=np.float32)

    with open(rsf_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3 and parts[1] in class_idx and parts[2] in class_idx:
                i = class_idx[parts[1]]
                j = class_idx[parts[2]]
                struct_mat[i][j] += 1
                struct_mat[j][i] += 1

    max_val = struct_mat.max()
    if max_val > 0:
        struct_mat /= max_val

    np.fill_diagonal(struct_mat, 1.0)
    return struct_mat

def build_semantic_matrix(class_list, model_name):
    """Generates LLM embeddings and builds the semantic similarity matrix."""
    print(f"\nLoading LLM: {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        token=hf_token
    )
    model = AutoModel.from_pretrained(
        model_name,
        trust_remote_code=True,
        token=hf_token,
        torch_dtype=torch.float16,
        quantization_config=bnb_config
    )
    model.eval()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Processing on: {device}")

    embeddings = []
    print(f"Embedding {len(class_list)} classes. This may take a minute...")

    for i, full_filename in enumerate(class_list):
        code = get_source_files(full_filename, source_files)[:4096]
        inputs = tokenizer(code, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().float().numpy()
        embeddings.append(emb)

        if (i + 1) % 25 == 0:
            print(f"  ...embedded {i + 1}/{len(class_list)} files")

    emb_array = np.array(embeddings, dtype=np.float32)
    sem_mat = cosine_similarity(emb_array).astype(np.float32)

    sem_mat = (sem_mat + 1.0) / 2.0
    return sem_mat


def run_arc(struct, semantic, class_list, alpha, k):
    """Fuses matrices, performs clustering, and writes the .rsf output."""
    combined_similarity = (alpha * struct) + ((1.0 - alpha) * semantic)
    distance_matrix = np.clip(1.0 - combined_similarity, 0.0, None)

    clusterer = AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage="complete")
    labels = clusterer.fit_predict(distance_matrix)

    output_path = Path(f"{output}/output_arc/alpha_{alpha}/arc_{k}")
    output_path.mkdir(parents=True, exist_ok=True)

    rsf_filename = output_path / f"tikadp-v1_ARC_{k}_clusters.rsf"

    with open(rsf_filename, "w", encoding="utf-8") as f:
        for cls_name, cluster_id in zip(class_list, labels):
            f.write(f"contain {cluster_id + 1} {cls_name}\n")

    print(f"Generated: {rsf_filename}")


if __name__ == "__main__":
    print("Starting Week 3 tasks - arc clustering")

    classes = get_classes(filtered_rsf)
    print(f"Total classes: {len(classes)}")

    print("\nBuilding structural matrix")
    struct_matrix = build_structural_matrix(filtered_rsf, classes)

    print("\nBuilding semantic matrix")
    sem_matrix = build_semantic_matrix(classes, embedding_model)

    alpha_values = [0.2, 0.4, 0.6, 0.8]
    k_values = range(2, 51)

    print("\nRunning Clustering Sweep...")
    for alpha in alpha_values:
        print(f"Generating clusters for Alpha = {alpha}")
        for k in k_values:
            run_arc(struct_matrix, sem_matrix, classes, alpha, k)

Starting Week 3 tasks - arc clustering
Total classes: 157

Building structural matrix

Building semantic matrix

Loading LLM: nomic-ai/nomic-embed-code...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Processing on: cuda
Embedding 157 classes. This may take a minute...
  ...embedded 25/157 files
  ...embedded 50/157 files
  ...embedded 75/157 files
  ...embedded 100/157 files
  ...embedded 125/157 files
  ...embedded 150/157 files

Running Clustering Sweep...
Generating clusters for Alpha = 0.2
Generated: /content/output_arc/alpha_0.2/arc_2/tikadp-v1_ARC_2_clusters.rsf
Generated: /content/output_arc/alpha_0.2/arc_3/tikadp-v1_ARC_3_clusters.rsf
Generated: /content/output_arc/alpha_0.2/arc_4/tikadp-v1_ARC_4_clusters.rsf
Generated: /content/output_arc/alpha_0.2/arc_5/tikadp-v1_ARC_5_clusters.rsf
Generated: /content/output_arc/alpha_0.2/arc_6/tikadp-v1_ARC_6_clusters.rsf
Generated: /content/output_arc/alpha_0.2/arc_7/tikadp-v1_ARC_7_clusters.rsf
Generated: /content/output_arc/alpha_0.2/arc_8/tikadp-v1_ARC_8_clusters.rsf
Generated: /content/output_arc/alpha_0.2/arc_9/tikadp-v1_ARC_9_clusters.rsf
Generated: /content/output_arc/alpha_0.2/arc_10/tikadp-v1_ARC_10_clusters.rsf
Generated: /con

In [10]:
!zip -r output_arc.zip /content/output_arc

  adding: content/output_arc/ (stored 0%)
  adding: content/output_arc/alpha_0.2/ (stored 0%)
  adding: content/output_arc/alpha_0.2/arc_13/ (stored 0%)
  adding: content/output_arc/alpha_0.2/arc_13/tikadp-v1_ARC_13_clusters.rsf (deflated 84%)
  adding: content/output_arc/alpha_0.2/arc_2/ (stored 0%)
  adding: content/output_arc/alpha_0.2/arc_2/tikadp-v1_ARC_2_clusters.rsf (deflated 85%)
  adding: content/output_arc/alpha_0.2/arc_23/ (stored 0%)
  adding: content/output_arc/alpha_0.2/arc_23/tikadp-v1_ARC_23_clusters.rsf (deflated 83%)
  adding: content/output_arc/alpha_0.2/arc_17/ (stored 0%)
  adding: content/output_arc/alpha_0.2/arc_17/tikadp-v1_ARC_17_clusters.rsf (deflated 84%)
  adding: content/output_arc/alpha_0.2/arc_19/ (stored 0%)
  adding: content/output_arc/alpha_0.2/arc_19/tikadp-v1_ARC_19_clusters.rsf (deflated 84%)
  adding: content/output_arc/alpha_0.2/arc_28/ (stored 0%)
  adding: content/output_arc/alpha_0.2/arc_28/tikadp-v1_ARC_28_clusters.rsf (deflated 83%)
  adding:

In [11]:
from google.colab import files
files.download('output_arc.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>